### Домашнее задание "PDA-1. Предобработка текстов с помощью Python":

Создать чат-бот-«барахольщик»: по продуктовому запросу он будет рекомендовать товары, по остальным запросам он будет отвечать «болталкой» (без фолбека).

### Задача:

- Файл с продуктами и описанием товаров расположен в папке "chatbot/ProductsDataset.csv"
- Обучить классификатор: продуктовый запрос vs. всё остальное (продуктовым можно считать запрос, который равен названию или описанию товара).
- Добавить логику поиска похожих товаров по продуктовому запросу.
- Вся логика должна быть завёрнута в метод get_answer(). Ответ на продуктовый запрос должен иметь вид "product_id title".

*Автотесты применяются к методу get_answer и проверяют несколько базовых сценариев:*

*assert(get_answer(“Юбка детская ORBY”).startswith(“58e3cfe6132ca50e053f5f82”))*
*assert(not get_answer(“Где ключи от танка”).startswith(“5”))*

### Критерии проверки:

**→ Обучен классификатор «товарный запрос vs. болталка»**

- Осуществлён препроцессинг текста (как минимум удаление знаков препинания, приведение к нижнему регистру, стемминг/лемматизация).
- Текст векторизирован любым из изученных способов (CountVectorizer, TfidfVectorizer, HashingVectorizer, Word2Vec).
- Выборка разделена на обучающую и валидационную.
- Обучен классификатор с расчётом метрик на валидации (любое семейство алгоритмов, но предпочтительнее просто логистическая регрессия).
- Модель сохранена и при применении загружается из pkl-файла (или аналога).


**→ Реализован поиск похожих товаров в контентной части бота**

- Все названия товаров свёрнуты в векторное представление Word2Vec (предобученном или обученном на исходном датасете).
- Построен индекс по названиям документов.
- Для товарных запросов реализован поиск в индексе (запрос также оборачивается Word2Vec, происходит проход в индекс).


**→ Реализована болталка**

- Все вопросы из датасета свёрнуты Word2Vec в векторное представление.
- Построен индекс по вопросам.
- На запрос в болталку происходит поиск ближайшего вопроса и возвращается ответ на этот вопрос.

Импорт библиотек:

In [ ]:
import os
import sys
import string
import annoy
import codecs
import setuptools
import pkg_resources

from pymorphy2 import MorphAnalyzer
from stop_words import get_stop_words
from gensim.models import Word2Vec

import numpy as np
from tqdm.notebook import tqdm
import pandas as pd

import pickle

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

Предобработаем ответы из файла chatbot/Otvety.txt: к каждому вопросу присоединим 1 ответ и запишем в файл на будущее. Это позволит нам сэкономить время и ресурсы при дальнейшем препроцессинге текста

In [3]:
question = None
written = False

#Мы идем по всем записям, берем первую строку как вопрос
# и после знака --- находим ответ
with codecs.open("prepared_answers.txt","w", "utf-8") as fout:
    with codecs.open("chatbot/Otvety.txt", "r", "utf-8") as fin:
        for line in tqdm(fin):
            if line.startswith("---"):
                written = False
                continue
            if not written and question is not None:
                fout.write(question.replace("\t", " ").strip() + "\t" + line.replace("\t", " "))
                written = True
                question = None
                continue
            if not written:
                question = line.strip()
                continue

0it [00:00, ?it/s]

Теперь нам нужно предобработать текст, чтобы обучить word2vec и получить эмбеддинги. Удаляем знаки препинания и делаем лемматизацию

In [18]:
def preprocess_txt(line):
    spls = "".join(i for i in line.strip() if i not in exclude).split()
    spls = [morpher.parse(i.lower())[0].normal_form for i in spls]
    spls = [i for i in spls if i not in sw and i != ""]
    return spls

In [8]:
sentences = []

morpher = MorphAnalyzer()
sw = set(get_stop_words("ru"))
exclude = set(string.punctuation)
c = 0

with codecs.open("chatbot/Otvety.txt", "r", "utf-8") as fin:
    for line in tqdm(fin):
        spls = preprocess_txt(line)
        sentences.append(spls)
        c += 1
        if c > 500000:
            break

0it [00:00, ?it/s]

In [9]:
# Обучим модель word2vec на наших вопросах
sentences = [i for i in sentences if len(i) > 2]
model = Word2Vec(sentences=sentences, vector_size=100, min_count=1, window=5)
model.save("w2v_model")

Теперь нам нужно сложить в индекс все вопросы. Используем библиотеку annoy. Проходимся по всем ответам, считаем, что вектор предложения - сумма word2vecов слов, которые входят в него (конечно же усредненная)

In [10]:
index = annoy.AnnoyIndex(100 ,'angular')

index_map = {}
counter = 0

with codecs.open("prepared_answers.txt", "r", "utf-8") as f:
    for line in tqdm(f):
        n_w2v = 0
        spls = line.split("\t")
        index_map[counter] = spls[1]
        question = preprocess_txt(spls[0])
        vector = np.zeros(100)
        for word in question:
            if word in model.wv:
                vector += model.wv[word]
                n_w2v += 1
        if n_w2v > 0:
            vector = vector / n_w2v
        index.add_item(counter, vector)
            
        counter += 1

index.build(10)
index.save('speaker.ann')

0it [00:00, ?it/s]

True

Теперь остается реализовать метод, который получит на вход вопрос и найдет ответ к нему!
Мы препроцессим вопрос, находим ближайший вопрос и выбираем ответ на ближайший вопрос.

In [12]:
def find_answer(question):
    preprocessed_question = preprocess_txt(question)
    n_w2v = 0
    vector = np.zeros(100)
    for word in preprocessed_question:
        if word in model.wv:
            vector += model.wv[word]
            n_w2v += 1
    if n_w2v > 0:
        vector = vector / n_w2v
    answer_index = index.get_nns_by_vector(vector, 1)
    return index_map[answer_index[0]]

Проверка функции выхова ответа:

In [15]:
find_answer("Юбка детская ORBY")

'Сама всегда удивляюсь, неужели им мёдом там намазано? Я терпеть не могу такие взгляды, но всем всегда интересно и посмотреть на маленького малыша. Больше чем уверена, что таких людей самих раздражало, когда в коляску к их детям заглядывали. Один раз видела, как ребёнок в коляске в торговом центре был прикрыт белой пеленкой.. \n'

In [16]:
find_answer("Где ключи от танка")

'Это тебе не одиночная игра, в онлайн играх нет кодов .. \n'

# Домашнее задание:

подрзуим данные о товарах:

In [22]:
products_df = pd.read_csv("chatbot/ProductsDataset.csv")
products_df.head(5)

,title,descrirption,product_id,category_id,subcategory_id,properties,image_links
0,Юбка детская ORBY,"Новая, не носили ни разу. В реале красивей чем...",58e3cfe6132ca50e053f5f82,22.0,2211,"{'detskie_razmer_rost': '81-86 (1,5 года)'}",http://cache3.youla.io/files/images/360_360/58...
1,Ботильоны,"Новые,привезены из Чехии ,указан размер 40,но ...",5667531b2b7f8d127d838c34,9.0,902,"{'zhenskaya_odezhda_tzvet': 'Зеленый', 'visota...",http://cache3.youla.io/files/images/360_360/5b...
2,Брюки,Размер 40-42. Брюки почти новые - не знаю как ...,59534826aaab284cba337e06,9.0,906,{'zhenskaya_odezhda_dzhinsy_bryuki_tip': 'Брюк...,http://cache3.youla.io/files/images/360_360/59...
3,Продам детские шапки,"Продам шапки,кажда 200р.Розовая и белая проданны.",57de544096ad842e26de8027,22.0,2217,"{'detskie_pol': 'Девочкам', 'detskaya_odezhda_...",http://cache3.youla.io/files/images/360_360/57...
4,Блузка,"Темно-синяя, 42 размер,состояние отличное,как ...",5ad4d2626c86cb168d212022,9.0,907,"{'zhenskaya_odezhda_tzvet': 'Синий', 'zhenskay...",http://cache3.youla.io/files/images/360_360/5a...


Сделает чат бот для ответов по товарам:

In [ ]:
ARTIFACT_DIR = "chatbot_artifacts"
INTENT_MODEL_PATH = os.path.join(ARTIFACT_DIR, "intent_model.pkl")
W2V_PATH = os.path.join(ARTIFACT_DIR, "w2v_bot.model")
CHAT_INDEX_PATH = os.path.join(ARTIFACT_DIR, "chat.ann")
CHAT_MAP_PATH = os.path.join(ARTIFACT_DIR, "chat_map.pkl")
PRODUCT_INDEX_PATH = os.path.join(ARTIFACT_DIR, "products.ann")
PRODUCT_MAP_PATH = os.path.join(ARTIFACT_DIR, "product_map.pkl")


if "morpher" not in globals():
    morpher = MorphAnalyzer()
if "sw" not in globals():
    sw = set(get_stop_words("ru"))
if "exclude" not in globals():
    exclude = set(string.punctuation)


_BOT = None


def _prepare_answers_file_if_missing():
    if os.path.exists("prepared_answers.txt"):
        return

    question = None
    written = False
    with codecs.open("prepared_answers.txt", "w", "utf-8") as fout:
        with codecs.open("chatbot/Otvety.txt", "r", "utf-8") as fin:
            for line in fin:
                if line.startswith("---"):
                    written = False
                    continue
                if not written and question is not None:
                    fout.write(question.replace("\t", " ").strip() + "\t" + line.replace("\t", " "))
                    written = True
                    question = None
                    continue
                if not written:
                    question = line.strip()


def _read_chat_pairs():
    _prepare_answers_file_if_missing()

    pairs = []
    with codecs.open("prepared_answers.txt", "r", "utf-8") as f:
        for line in f:
            parts = line.strip().split("\t", 1)
            if len(parts) == 2 and parts[0].strip() and parts[1].strip():
                pairs.append((parts[0].strip(), parts[1].strip()))
    return pairs


def _load_products():
    df = pd.read_csv("chatbot/ProductsDataset.csv")
    # В датасете колонка description записана с опечаткой: descrirption.
    df = df.rename(columns={"descrirption": "description"})
    df["title"] = df["title"].fillna("").astype(str)
    df["description"] = df["description"].fillna("").astype(str)
    df["product_id"] = df["product_id"].astype(str)
    return df


def _sentence_vector(tokens, w2v_model):
    vector = np.zeros(w2v_model.vector_size, dtype=np.float32)
    n_w2v = 0
    for word in tokens:
        if word in w2v_model.wv:
            vector += w2v_model.wv[word]
            n_w2v += 1
    if n_w2v > 0:
        vector = vector / n_w2v
    return vector


def _build_and_save_artifacts():
    os.makedirs(ARTIFACT_DIR, exist_ok=True)

    chat_pairs = _read_chat_pairs()
    products = _load_products()

    # 1) Классификатор: товарный запрос vs болталка.
    positive_texts = products["title"].tolist() + products["description"].tolist()
    negative_texts = [q for q, _ in chat_pairs]

    X_raw = positive_texts + negative_texts
    y = np.array([1] * len(positive_texts) + [0] * len(negative_texts))

    X_clean = [" ".join(preprocess_txt(text)) for text in X_raw]

    X_train_raw, X_val_raw, y_train, y_val = train_test_split(
        X_clean,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
    X_train = vectorizer.fit_transform(X_train_raw)
    X_val = vectorizer.transform(X_val_raw)

    clf = LogisticRegression(max_iter=1000, class_weight="balanced")
    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_val)
    print("Intent classifier accuracy:", round(accuracy_score(y_val, y_pred), 4))
    print(classification_report(y_val, y_pred, digits=4))

    with open(INTENT_MODEL_PATH, "wb") as f:
        pickle.dump({"vectorizer": vectorizer, "clf": clf}, f)

    # 2) Word2Vec для поиска похожих товаров и болталки.
    corpus = []
    corpus.extend(preprocess_txt(q) for q, _ in chat_pairs)
    corpus.extend(preprocess_txt(t) for t in products["title"].tolist())
    corpus.extend(preprocess_txt(d) for d in products["description"].tolist())
    corpus = [tokens for tokens in corpus if len(tokens) > 0]

    w2v = Word2Vec(
        sentences=corpus,
        vector_size=100,
        window=5,
        min_count=1,
        sg=1,
        workers=4,
        epochs=15
    )
    w2v.save(W2V_PATH)

    # 3) Индекс болталки по вопросам.
    chat_index = annoy.AnnoyIndex(w2v.vector_size, "angular")
    chat_map = {}

    for idx, (question, answer) in enumerate(chat_pairs):
        vec = _sentence_vector(preprocess_txt(question), w2v)
        chat_index.add_item(idx, vec)
        chat_map[idx] = answer

    chat_index.build(20)
    chat_index.save(CHAT_INDEX_PATH)
    with open(CHAT_MAP_PATH, "wb") as f:
        pickle.dump(chat_map, f)

    # 4) Индекс товаров по названиям.
    product_index = annoy.AnnoyIndex(w2v.vector_size, "angular")
    product_map = {}
    exact_title_map = {}

    for idx, row in products.reset_index(drop=True).iterrows():
        title = str(row["title"])
        title_norm = " ".join(preprocess_txt(title))
        vec = _sentence_vector(preprocess_txt(title), w2v)

        product_index.add_item(idx, vec)
        product_map[idx] = {
            "product_id": row["product_id"],
            "title": title
        }

        if title_norm:
            exact_title_map[title_norm] = idx

    product_index.build(20)
    product_index.save(PRODUCT_INDEX_PATH)

    with open(PRODUCT_MAP_PATH, "wb") as f:
        pickle.dump({"product_map": product_map, "exact_title_map": exact_title_map}, f)


def _load_or_build_bot():
    need_build = not all(
        os.path.exists(path)
        for path in [
            INTENT_MODEL_PATH,
            W2V_PATH,
            CHAT_INDEX_PATH,
            CHAT_MAP_PATH,
            PRODUCT_INDEX_PATH,
            PRODUCT_MAP_PATH,
        ]
    )

    if need_build:
        _build_and_save_artifacts()

    with open(INTENT_MODEL_PATH, "rb") as f:
        intent_data = pickle.load(f)
    vectorizer = intent_data["vectorizer"]
    clf = intent_data["clf"]

    w2v = Word2Vec.load(W2V_PATH)

    chat_index = annoy.AnnoyIndex(w2v.vector_size, "angular")
    chat_index.load(CHAT_INDEX_PATH)
    with open(CHAT_MAP_PATH, "rb") as f:
        chat_map = pickle.load(f)

    product_index = annoy.AnnoyIndex(w2v.vector_size, "angular")
    product_index.load(PRODUCT_INDEX_PATH)
    with open(PRODUCT_MAP_PATH, "rb") as f:
        product_data = pickle.load(f)

    return {
        "vectorizer": vectorizer,
        "clf": clf,
        "w2v": w2v,
        "chat_index": chat_index,
        "chat_map": chat_map,
        "product_index": product_index,
        "product_map": product_data["product_map"],
        "exact_title_map": product_data["exact_title_map"],
    }


def _is_product_query(question, bot):
    question_norm = " ".join(preprocess_txt(question))
    if not question_norm:
        return False

    x = bot["vectorizer"].transform([question_norm])
    pred = bot["clf"].predict(x)[0]
    return bool(pred)


def _answer_product(question, bot):
    question_tokens = preprocess_txt(question)
    question_norm = " ".join(question_tokens)

    if question_norm in bot["exact_title_map"]:
        idx = bot["exact_title_map"][question_norm]
    else:
        vec = _sentence_vector(question_tokens, bot["w2v"])
        idx = bot["product_index"].get_nns_by_vector(vec, 1)[0]

    item = bot["product_map"][idx]
    return f"{item['product_id']} {item['title']}"


def _answer_chat(question, bot):
    vec = _sentence_vector(preprocess_txt(question), bot["w2v"])
    idx = bot["chat_index"].get_nns_by_vector(vec, 1)[0]
    return bot["chat_map"][idx]


def get_answer(question):
    global _BOT

    if _BOT is None:
        _BOT = _load_or_build_bot()

    if _is_product_query(question, _BOT):
        return _answer_product(question, _BOT)
    return _answer_chat(question, _BOT)


# Базовые автотесты из задания
assert get_answer("Юбка детская ORBY").startswith("58e3cfe6132ca50e053f5f82")
assert not get_answer("Где ключи от танка").startswith("5")
print("Smoke tests passed")

Intent classifier accuracy: 0.9845
              precision    recall  f1-score   support

           0     0.9961    0.9874    0.9917    232669
           1     0.8198    0.9371    0.8745     14219

    accuracy                         0.9845    246888
   macro avg     0.9080    0.9622    0.9331    246888
weighted avg     0.9860    0.9845    0.9850    246888

Smoke tests passed


In [20]:
get_answer("Юбка детская ORBY")

'58e3cfe6132ca50e053f5f82 Юбка детская ORBY'

In [21]:
get_answer("Где ключи от танка")

'Ща проверим!.'